# Release notes

This page lists the main changes in each version of onceover, newest first. The date of a version is the date of its tag in the repository; packages are published to PyPI and crates.io from the tag. Work that is merged but not yet tagged is collected under Unreleased.


## Unreleased

Work merged since version 0.1.4, not yet tagged.

### Hierarchies, declared structure, and the time axis

- Hierarchy chains collapse. Pairwise functional dependencies and rollups over different column sets that form one hierarchy — each level determining the next — are consolidated under a single lead finding that states the levels in order, with each adjacent mapping's row support and every member relation in its evidence. The restatements move to `grouped_findings` and spend no selection slots. Columns that determine each other count as one level; a branching or merging structure is not a chain and is not collapsed.
- Two new hint keys. `hierarchies` declares a known hierarchy (level columns ordered finest to coarsest): the expectations pass verifies every adjacent mapping exactly, a hierarchy that holds is stated once with the inferred restatements grouped under it, and a hierarchy that does not hold is stated once as a deviation naming every broken mapping. `exclude_columns` removes columns before profiling starts — no card, no findings — with the exclusion and its user-declared source stated in a `column_exclusions` coverage entry.
- The time axis is excluded from target roles by structure. The column the frame model assembles as the report's time axis no longer stays target-eligible when it carries an inferred role such as measure (a bare week integer was predictable as a target), and the series pass does not profile it as a measure. Coverage states the exclusion; a `roles` entry for the axis column overrides it.

### Constraint checks

- The multi-term identity scan now also finds exact decompositions of four to eight parts — a total that is the sum of its components — by a components-first search: candidate parts are the columns lying between zero and the target on at least 99% of sampled rows, and parts are subtracted from the target on the sample until the residual vanishes. Candidates are verified on every row under the same exactness standard as the existing forms, and the scan keeps its tier behaviour (off at `quick`).
- The order-constraints scan now states the complementary case. When a pair's less-violated direction fails on at least 20 rows and at least 5% of the evaluated rows while the reverse direction fails at least twice as often, and the two columns track each other on a comparable scale, the report carries an `order_violation` finding with the violating row count and share, the reverse count, and the violating rows in the evidence. Neither direction holds for such a pair; the statement names the failure instead of leaving the pair silent.
- On a frame with an entity × time grain, the series scan now reports one-way flags: a boolean or 0/1 flag that switches off-to-on within entities and never on-to-off is stated as a `one_way_flag` finding, with the number of entities showing an onset and, where a measure is zero before onset and positive after for the same entities, the lag from onset to first activity as a median and middle half. Where the frame has no such grain, the coverage record says the check does not apply.

### Panel coverage

- New `panel_coverage` detector, in the default set: per-period coverage on the frame's time axis. Each observed period's row count — and its distinct entity count, where the grain or a single identifier column provides an entity axis — is compared with the median period's; a period below 75% of the typical count (and short by at least 4 rows, or at least 2 entities) is reported as a `partial_period` finding (category `gaps`) stating its position (first, last, or interior), its counts, and the typical counts, with the per-period table in the evidence. The detector registry grows to 36 detectors, 29 in the default set. A frame with no time axis records the pass as skipped in coverage.
- Per-period coverage now also reads measure volume. Each period's totals for the frame's principal measures — the frame model's measure-role columns, ranked by non-null observation count and capped at 8 — are reported in the coverage entry and in each `partial_period` finding's evidence, and a period whose principal-measure total falls below 75% of that measure's median period total is reported, so a period holding the usual rows and entities while the measured activity in it collapses is stated. A period is stated once however many of its volumes fall short. A measure whose median total is not positive, or that carries negative values, is totalled but never flags: its sum is a level, not a volume.

### Time-series passes

- Trend statements carry a significance test. A series trend is now stated only when the slope clears a t-test deflated by the residuals' lag-1 autocorrelation (effective sample size n(1-rho)/(1+rho)) at p < 0.001, alongside the existing line-fit gates; the evidence gains `slope_p_value`, `residual_autocorrelation`, and `effective_points`. A random walk, whose line fit explains much of its variance, is no longer stated as a trend.
- Breaks findings carry a located range. `level_shift`, `trend_change`, and `variance_change` findings gain `breakpoint_range_start` and `breakpoint_range_end` (with `_raw` forms): every split whose two-segment squared error lies within 3.84 residual variances of the best split's, read off the detection scan's own cost curve. The claim states the range and keeps the best split as the point estimate.
- Weekly axes are recognized on their own terms. The frame model assembles a year and ISO week part pair as the Monday of each week, records a lone week-number column as a low-confidence candidate axis, and describes an axis with a seven-day modal step as weekly. On a weekly axis the series pass tests the annual cycle at lag 52 once the span reaches two years.
- The series pass reads panels per entity. On an entity × time grain its trend and level-break checks also run per entity on the principal measure (at least 12 points for a trend, at least 24 for a break), and the findings name the entities carrying each pattern with the tested count. An aggregate trend carried by a minority of entities, or by none, says so in its statement.

### Column labels

- A new hint key, `labels`, maps a column name to a display label. Claims and interpretations then name the column by its label, so a report can be read by people who do not know the raw column names. The machine-readable record is unchanged: subjects, specifications, evidence metrics, and column cards keep the raw column names, and finding ids — derived from the canonical statement, not the claim text — are identical with and without labels. A label that names a different column of the table, one label declared for two columns, an empty label, or a label for an unknown column is not applied; the raw name stays in the prose, and the report's `labels` coverage entry states each case.

### Comparing two versions of a dataset

- `report.diff(other)` now compares the column cards as well as the findings. Under `columns` it states the columns the other profile added and the ones it removed, the columns whose card dtype changed (retyped, stated distinctly from a removal plus an addition), and, for the columns in both with an unchanged dtype, the card quantities that moved — rows, nulls and their share, distinct values, and for numeric columns the mean, minimum, and maximum the card carries — each with its before and after values, the absolute change, and the relative change where one is meaningful. The compared quantities are computed by the column pass on the full table, so profiles taken at different budgets do not manufacture card differences. A renamed column reads as a removal plus an addition; row-level changes are not compared.

### Panel dynamics

- On a frame with an entity × time grain, the series scan now reports entry and exit blocks: a period in which the entities first appearing (or last appearing) number at least 5, at least four times the median per-period count for that direction, and at least 2% of all entities is stated as an `entry_block` (or `exit_block`) finding, with the period, the count and its share of all entities, and the per-period entry and exit counts for the whole axis in the evidence. The first period cannot be an entry block and the last period cannot be an exit block: every founding entity first appears in the first period, and entities present in the last period are still under observation, not departed.
- Where a measure exists, the scan also dates first activity. Each entity's first active period on the principal measure (the measure with the most entities ever active; active means non-zero for a numeric measure, true for a boolean, present otherwise) is set against its first observed period. When at least 3 entities start late and late starters are at least 10% of the entities that ever become active, a `first_activity` finding states how many start in their first observed period, how many start late with the lag as a median and middle half, and how many never become active — with every entity's first, last, and first-activity periods in the evidence. Where the frame has no such grain, the coverage record says the check does not apply.

### Group contrast

- A declared subset of the rows can now be profiled against the rest of the table. The new `contrast` hint names the group — a boolean column's true rows, or the rows where a column equals a value — and the new hint-gated `group_contrast` detector compares the two populations on every other column: means and medians with an effect size for numeric columns, level shares for categorical columns, the true share for boolean columns. A difference is stated only when the sizes support it (at least 10 known values on each side, an effect of at least 0.5 pooled standard deviations or a share difference of at least 0.10, and at least 3 standard errors), findings are ranked by effect size and capped at 10, and the coverage entry records the group, both row counts, and how many contrasts cleared the bars. A run that declares no contrast is unchanged.

### Panel concentration

- On a frame with an entity × time grain, the series scan now also reads where a measure's mass sits among entities, in a `panel_concentration` coverage entry. Type `concentration_over_time` states the share of each period's total held by the top group of entities (the largest 1% of entities, at least one) counting only periods in which at least half of the entities are active, when that share is high in level — a median period at or above 30% — or rising materially, the last third of periods above the first third by at least 10 share points and at least 20%. Type `outlier_entity` states entities whose total of a measure over all periods is at least 10 times the typical entity's total and above the upper quartile plus three interquartile ranges, naming the entities with their totals, multiples, and shares. Both readings need at least 20 entities; where the frame has no such grain, the coverage record says the check does not apply. The whole-column concentration readings are unchanged: they pool the periods and cannot make either statement.

### Report verification

- The report verifier now recounts duplicate-encoding and rollup statements in full. It previously recounted only the distinct value pairs and the complete rows for those statements, so a claim of either kind edited after the fact could pass verification silently. It now also recounts the levels on each side, the rows breaking each direction's majority mapping, and the exceptions; for an exact duplicate encoding — a bijection — it recomputes whether the pairing is one-to-one from the violations in both directions, and a rollup is recounted as the mapping it states, finer to coarser. Every exact-relation claim in a report is now one the verifier checks against the data.

### Views coverage

- The views coverage entry now states its dropped columns under two separate wordings: the columns dropped past the 64-column cap, with their count, and the columns dropped at extraction (unsupported type or no non-null values), with their count. Extraction-dropped columns were never eligible, so they were never subject to the cap; the entry no longer attributes them to it.

### Column types

- Duration columns are analysed as numbers, in seconds. Previously they were accepted but opaque: counted for rows and nulls, with no statistics and no detector examining them. A duration column now receives the numeric card, is eligible for the measure role, and is visible to every pass that consumes numeric columns. Values are converted to float at extraction (the stored tick count divided by the ticks per second at the column's time unit), so spans beyond float precision are not retained exactly.
- Time-of-day columns are counted but not analysed, and coverage now says so. A time of day is a circular quantity — 23:00 is closer to 01:00 than to 12:00 — so the order statistics and measures the analysed kinds receive would mislead. The extraction coverage entry names the time-of-day columns and states the treatment, instead of leaving them silently opaque.
### Breaks coverage

- The breaks pass now states the series its series cap leaves unscanned in depth as a finding, not only in coverage: when eligible series exceed the cap, one `unscanned_series` finding (category `gaps`) names the unscanned series and carries each one's prescore in the evidence — every eligible series is prescored over its full span before the cap binds, so the finding states what the prescore saw as well as what the in-depth scan skipped. The scanned set is still chosen by prescore, the coverage entry names the same series, and nothing changes when the cap is not reached.
### Order violations for part-against-whole pairs

- The order pass now states a part exceeding its whole in bulk. A part sits far from its total by construction, so the systematic order-violation statement's comparable-scale gate (median row distance at most 0.2) could never pass for a part-against-whole pair, and the identity scan states a decomposition only when it holds on at least 99.9% of rows — which bulk corruption breaks. A corrupted part was therefore stated by no pass. The `order_violation` finding now also fires, with `violation_kind: part_whole` in the evidence, when a direction's violations number at least 20 and between 5% and 40% of the evaluated rows and one column is a component of the other: by identity, when the whole minus the part equals a sum of one to three other columns on at least 99.9% of the rows where the order holds, with the rest non-negative; or by containment, when both columns respect the shared zero floor and the slack reaches it on at least 5% of the held rows but fewer than half of them. Where an exact sum identity states the decomposition, this statement stays silent for the pair — the same failure is not stated twice.


## Version 0.1.4 — 9 November 2026

Version 0.1.4 was tagged and published on 9 November 2026. The main changes since 0.1.3 follow.

### Command line and HTML report

- Installing the package also installs an `onceover` command: it profiles a file from the shell and prints the digest by default, or writes the report as JSON or markdown or renders it as HTML. Several files given together are read as a set of tables — the `profile_many` route. `python -m onceover` is equivalent, and a usage page in this book documents the command.
- `report.to_html()` renders a report as one self-contained HTML file: the digest, the findings grouped by category, and a chart for each numeric column, with the charting library embedded so the file opens offline. It needs the `html` extra.

### The book and the benchmark corpus

- The benchmark corpus entry for the satellites table now runs the order-constraints check; the statements it produces were verified against the raw data before the new expectations were recorded. Machine-verified corpus expectations stand at 93 of 93.
- Three methodology pages work one check through on real data: tested associations, including which strong pairs the Simpson check does not examine; values piled up at one point of a column (boundary mass); and order constraints between columns.
- The glossary defines account, deadline, digest and source — four terms the report and this book used without defining.

### Column types

- Decimal columns are analysed as numbers. Previously they were accepted but opaque: counted for rows and nulls, with no statistics and no detector examining them. A decimal column now receives the numeric card, is eligible for the measure role, and is visible to every pass that consumes numeric columns. Values are converted to float at extraction (the stored scaled integer divided by 10^scale), so digits beyond float precision are not retained.


## Version 0.1.3 — 7 November 2026

Version 0.1.3 was tagged and published on 7 November 2026. The main changes since 0.1.2 follow.

### The report

- The report schema moved from 0.2 to 0.7. The table at the end of this page lists each step and what it changed.
- `report.to_digest(chars)` returns a digest of the report that fits a stated character budget.
- `report.diff(other)` reports the findings added, removed or changed between two profiles of a table, matched by stable id, plus coverage status changes.
- `report.to_json(max_chars=...)` returns a size-budgeted JSON view: findings are kept in full in rank order until the budget is reached and collapse to index entries after that, and a `budget` record in the output states what was cut.
- A strong association statement is qualified when the association reverses inside every subgroup of a category column (a Simpson reversal) and the combined within-subgroup effect is significant. The qualification states the per-subgroup numbers; the original statement stays in the report.
- Predictive statements name the columns that drive the prediction, not every input column.
- Statements are checkable against the data: a categorical outlier statement lists every row its share counts, and findings carry the values a reader needs to reproduce them.

### Detectors

- Ordered partitions: regime trees over a time axis or a dimension, split where the level or the trend changes.
- Boundaries: a value piled at a column's observed minimum or maximum, the shape a cap or a floor on recording would leave.
- Missingness: each column missing on at least 1% of rows, the pattern behind the missingness when one is clear, and blocks of columns that go missing on the same rows.
- Content: what a text column actually holds — numbers, dates or booleans stored as text, recognisable formats such as email addresses and URLs, and labels that differ only in case or spacing.
- Order constraints between pairs of columns: one column's values preceding another's in every row, such as a start column before an end column.
- Near-duplicate rows (beyond what chance coincidence on the compared columns explains), values swapped between rows, and rows sitting off a line.
- Most of the rest of the battery also dates from this cycle: breaks in level, trend and variance over time, with transient excursions; associations computed separately per period; long-format (pivot) structure; multi-term arithmetic identities; subset-conditional and categorical surprises; cross-sectional discontinuities; duplicate encodings; interaction strength between predictors; target-aware shape decomposition; digit fingerprints; sentinel values; hotspots; granularity changes in float columns; drift along a time axis; out-of-order appended blocks (backfills).

### Inputs and budgets

- `profile()` accepts a Polars LazyFrame or a file path (Parquet, CSV or IPC) as well as a DataFrame. A large source is profiled from a stratified sample of at most 200,000 rows by default, and the report states whether a number rests on the full data or on the sample, and how large the source is. `exact=True` collects the whole source regardless of the cap, and `sample_seed` fixes the seed the sample is drawn with, so the same source, cap and seed give the same sample and the same report.
- `profile_many()` reads a set of tables together and reports candidate foreign keys: column pairs from different tables whose distinct non-null values are fully contained in a unique column on the other side. Candidates whose column names agree form a strong tier; the rest are reported too, each with a chance score for the containment happening by accident. Partial containment into a unique parent is reported as an overlap with exact orphan counts. The result is a separate `MultiTableReport`; the individual tables are not profiled.
- A caller can declare hints: the time axis, an ordering, gap-free columns, and which columns are dimensions and which are measures.
- Frames with narrow-integer, decimal, struct or fixed-size-array columns are read in instead of failing at ingestion.
- A time budget is also checked inside the five longest passes (the dependence screen, hotspots, discontinuities, surprises and duplicate encodings). A pass stopped mid-way reads `capped` in coverage and states how much of its search it examined. Work before the first check (extraction, column cards, the frame model) cannot be interrupted, so a very short budget can still be exceeded.
- Coverage has an entry for every registered detector. A detector that did not run says why: not enabled, opt-in, quick tier, or deadline reached.
- `profile(timing=True)` records per-pass wall time as `elapsed_ms` on each coverage entry. Findings are identical either way.
- Passes with more candidates than their budget allows choose by a cheap pre-score instead of by column order: ordered partitions, hotspots and the breaks scan.
- Five detectors that never fired on the benchmark corpus (dates, pivot, period associations, drift, granularity) are opt-in rather than default. Report size, not time, is the binding limit for the agents that read these reports.
- The hotspots and dependence searches stream their enumeration, so a wide table does not need its full search space held in memory.
- The discontinuities pass reuses the axis data it computes across targets. On the Taiwanese bankruptcy data the pass took 8,770 ms where it took 13,055 ms before, with identical reports.
- The supported Polars range remains 1.28 and later, as well as 2.x. A wheel test exercises the oldest supported 1.x release; a failure there was traced to a date construction inside one test and fixed in the test, with no change to the supported range.

### The evidence behind the reports

- The benchmark harness stopped counting documented misses as passes. Machine-verified expectations stand at 89 of 89; documented cases are recorded as found (18), partial (4) or missed (10). A planted-signal battery with pinned baselines was added beside it.
- Correctness fixes preserve exact values end to end and bound the work a profile does.
- Functional dependencies are scored and selected by their chance-corrected information (reliable fraction of information), so a determinant with nearly one value per row no longer ranks as a strong dependency. Composite determinants that look unique only on the screening sample are verified on all rows instead of dismissed.
- The terminal-digit test compares a column with the digits its own values can end in (a column of even numbers with the five even digits) and runs only when the central 90% of the values spans at least three cycles of the last digit.
- An appendix to this book evaluates foundation-model screeners (Kumo and Timestral) against plain baselines, with each result stated next to its baseline.


## Version 0.1.2 — 27 October 2026

Report schema 0.2.

- onceover works with Polars 1.28 and later, as well as Polars 2.x. Before this version it required Polars 2.x.


## Version 0.1.1 — 27 October 2026

Report schema 0.2.

- New views detector: a greedy search that groups columns into views, subsets whose joint behaviour is simpler than the whole table's. The detector is opt-in.
- Correctness fixes in the dependence screen and the views search: a marginal distribution, merges that added nothing, selector collisions, and leave-one-out assignment. Budget use is reported more clearly.


## Version 0.1.0 — 26 October 2026

First release. Report schema starts at 0.2.

- A profile of a Polars DataFrame returns column summaries, relationship findings with the evidence beside each claim, and a record of what was searched (coverage).


## Report schema history

Every tagged version before 0.1.3 reports schema 0.2. Each later step is in version 0.1.3.

| Report schema | Version | What changed in the report |
| --- | --- | --- |
| 0.2 | 0.1.0 | Evidence-first report: findings carry their evidence beside the claim. Per-type evidence payloads were added within 0.2. |
| 0.3 | 0.1.3 | Every finding carries facets — scope, object, evidence, conditionality, temporality and status as fields — and its category is derived from the facets by a fixed table. |
| 0.4 | 0.1.3 | The frame model: each column's semantic role and the assembled time axis are computed before the detectors run and included in the report. |
| 0.5 | 0.1.3 | An account block states the table's grain, key and time axis. The digest re-ranks statements and demotes restatements. |
| 0.6 | 0.1.3 | Verified relationship families collapse to one lead statement; the remaining members are listed in `grouped_findings`. |
| 0.7 | 0.1.3 | Associations are tested pair by pair: the report lists every tested pair with its q-value, weak pairs collapse into one digest line, and `report.associations()` returns the full table. A Simpson-reversal qualification for strong statements was added within 0.7. |
